# 24 - Repaso de probabilidad, probabilidad condicional y teorema de Naive Bayes en R

## Proposito de la sesion

En esta sesion vas a:

1. Repasar experimento aleatorio, espacio muestral y eventos.
2. Conectar probabilidad teorica con frecuencia empirica.
3. Entender union, interseccion, complemento e independencia.
4. Calcular probabilidad condicional con datos reales.
5. Usar el teorema de Bayes para invertir informacion.
6. Construir un clasificador Naive Bayes paso a paso en R.
7. Cerrar con ejercicios que exigen pensar, no solo copiar codigo.


In [ ]:
# Configuracion minima para la sesion.
set.seed(2026)
options(scipen = 999)

# Titanic viene en R base como tabla agregada.
titanic_tab <- as.data.frame(Titanic)
head(titanic_tab)


## 1) Repaso rapido de probabilidad

Una **probabilidad** mide que tan plausible es un evento.

Tres ideas basicas:

- Un **experimento aleatorio** produce resultados inciertos.
- El **espacio muestral** contiene todos los resultados posibles.
- Un **evento** es un subconjunto del espacio muestral.

Si el experimento es lanzar un dado justo, el espacio muestral es:

$$S = \{1,2,3,4,5,6\}$$

Si todos los resultados son igual de probables, entonces:

$$P(A) = \frac{|A|}{|S|}$$

Tambien podemos aproximar probabilidades con frecuencias empiricas al simular muchas veces.


In [ ]:
# Espacio muestral de un dado justo.
espacio <- 1:6

# Dos eventos de ejemplo.
A <- c(2, 4, 6)   # sale numero par
B <- c(5, 6)      # sale un numero mayor que 4

# Probabilidades teoricas.
p_A_teorica <- length(A) / length(espacio)
p_B_teorica <- length(B) / length(espacio)

# Aproximacion empirica por simulacion.
lanzamientos <- sample(espacio, size = 10000, replace = TRUE)
p_A_empirica <- mean(lanzamientos %in% A)
p_B_empirica <- mean(lanzamientos %in% B)

data.frame(
  evento = c('par', 'mayor_que_4'),
  teorica = c(p_A_teorica, p_B_teorica),
  empirica = c(p_A_empirica, p_B_empirica)
)


## 2) Union, interseccion y complemento

Con eventos podemos combinar informacion:

- **Union**: ocurre `A` o ocurre `B`.
- **Interseccion**: ocurren `A` y `B` al mismo tiempo.
- **Complemento**: no ocurre `A`.

Reglas utiles:

$$P(A^c) = 1 - P(A)$$

$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

La resta evita contar dos veces los elementos compartidos por ambos eventos.


In [ ]:
# Operaciones entre eventos sobre el dado.
union_AB <- union(A, B)
inter_AB <- intersect(A, B)
comp_A <- setdiff(espacio, A)

p_union <- length(union_AB) / length(espacio)
p_inter <- length(inter_AB) / length(espacio)
p_comp_A <- length(comp_A) / length(espacio)

# Verificacion de la formula de la union.
p_union_formula <- p_A_teorica + p_B_teorica - p_inter

list(
  union = union_AB,
  interseccion = inter_AB,
  complemento_de_A = comp_A,
  prob_union_directa = p_union,
  prob_union_formula = p_union_formula,
  prob_complemento_A = p_comp_A
)


## 3) Frecuencia empirica y ley de los grandes numeros

En la practica no siempre conocemos la probabilidad teorica.
Por eso estimamos probabilidades observando datos.

La intuicion de la **ley de los grandes numeros** es que, si repetimos muchas veces el mismo experimento, la frecuencia relativa se estabiliza cerca de la probabilidad real.

No significa que cada bloque pequeno de observaciones sea "bonito" o equilibrado.
Significa que la tendencia global se vuelve mas estable cuando el numero de repeticiones crece.


In [ ]:
# Evolucion de la probabilidad empirica de obtener un numero par.
set.seed(2026)

n <- 500
corrida <- sample(espacio, size = n, replace = TRUE)
proporcion_par <- cumsum(corrida %in% A) / seq_len(n)

plot(
  proporcion_par,
  type = 'l',
  lwd = 2,
  col = 'steelblue',
  ylim = c(0, 1),
  xlab = 'numero de lanzamientos',
  ylab = 'proporcion acumulada',
  main = 'Convergencia empirica de P(numero par)'
)
abline(h = p_A_teorica, col = 'firebrick', lty = 2, lwd = 2)
legend(
  'topright',
  legend = c('empirica', 'teorica'),
  col = c('steelblue', 'firebrick'),
  lty = c(1, 2),
  lwd = 2,
  bty = 'n'
)


## 4) Independencia

Dos eventos `A` y `B` son **independientes** si conocer uno no cambia la probabilidad del otro.

Formalmente:

$$P(A \cap B) = P(A)P(B)$$

Si esta igualdad no se cumple, entonces hay dependencia.

La independencia es importante porque luego aparecera como el supuesto central de **Naive Bayes**.


In [ ]:
# Simulacion de independencia con dos monedas justas.
set.seed(2026)

moneda_1 <- sample(c('cara', 'cruz'), size = 20000, replace = TRUE)
moneda_2 <- sample(c('cara', 'cruz'), size = 20000, replace = TRUE)

evento_A <- moneda_1 == 'cara'
evento_B <- moneda_2 == 'cara'

c(
  P_A = mean(evento_A),
  P_B = mean(evento_B),
  P_interseccion = mean(evento_A & evento_B),
  P_A_por_P_B = mean(evento_A) * mean(evento_B)
)


## 5) Probabilidad condicional

La **probabilidad condicional** responde preguntas del tipo:

> "Si ya se que ocurrio `B`, que tan probable es `A`?"

Su definicion es:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

siempre que `P(B) > 0`.

Esta formula cambia el universo de referencia: ya no miramos todos los casos, sino solo los casos donde `B` ocurrio.

Para que esto sea mas concreto usaremos el dataset `Titanic`.


In [ ]:
# Probabilidad total de sobrevivir en Titanic.
p_survive <- with(subset(titanic_tab, Survived == 'Yes'), sum(Freq)) / sum(titanic_tab$Freq)
cat('P(Survived = Yes):', round(p_survive, 4), '

')

# Probabilidades condicionales de supervivencia por sexo y por clase.
tab_sex_surv <- xtabs(Freq ~ Sex + Survived, data = titanic_tab)
tab_class_surv <- xtabs(Freq ~ Class + Survived, data = titanic_tab)

survival_by_sex <- prop.table(tab_sex_surv, margin = 1)
survival_by_class <- prop.table(tab_class_surv, margin = 1)

round(survival_by_sex, 3)
round(survival_by_class, 3)


## 6) Leer bien una condicion

Estas dos preguntas **no** son iguales:

- `P(Survived = Yes | Sex = Female)`
- `P(Sex = Female | Survived = Yes)`

La primera pregunta condiciona en el sexo.
La segunda condiciona en haber sobrevivido.

Cambiar la condicion cambia el denominador, y por lo tanto cambia la interpretacion.
Ese error aparece con mucha frecuencia cuando se estudia Bayes por primera vez.


In [ ]:
# Comparacion de direcciones de condicionamiento.
p_yes_given_female <- survival_by_sex['Female', 'Yes']
p_female_given_yes <- prop.table(xtabs(Freq ~ Survived + Sex, data = titanic_tab), margin = 1)['Yes', 'Female']

cat('P(Survived = Yes | Sex = Female):', round(p_yes_given_female, 4), '
')
cat('P(Sex = Female | Survived = Yes):', round(p_female_given_yes, 4), '

')

# Visualizacion rapida de supervivencia condicionada.
par(mfrow = c(1, 2))
barplot(
  survival_by_sex[, 'Yes'],
  col = c('tomato', 'steelblue'),
  ylim = c(0, 1),
  main = 'P(sobrevive | sexo)',
  ylab = 'probabilidad'
)
barplot(
  survival_by_class[, 'Yes'],
  col = c('goldenrod', 'darkseagreen', 'steelblue', 'slategray'),
  ylim = c(0, 1),
  main = 'P(sobrevive | clase)',
  ylab = 'probabilidad'
)
par(mfrow = c(1, 1))


## 7) Teorema de Bayes

El teorema de Bayes permite **invertir** condicionamientos.

$$P(A \mid B) = \frac{P(B \mid A)P(A)}{P(B)}$$

Lectura practica:

- `P(A)` es la probabilidad previa o **prior**.
- `P(B | A)` es cuan compatible es la evidencia `B` con la hipotesis `A`.
- `P(A | B)` es la probabilidad actualizada despues de observar `B`.

Bayes no es magia: es una reorganizacion algebraica de la probabilidad condicional.


In [ ]:
# Verificacion numerica de Bayes con Titanic.
p_female <- sum(tab_sex_surv['Female', ]) / sum(tab_sex_surv)
p_yes <- sum(tab_sex_surv[, 'Yes']) / sum(tab_sex_surv)
p_yes_given_female <- tab_sex_surv['Female', 'Yes'] / sum(tab_sex_surv['Female', ])

p_female_given_yes_directa <- tab_sex_surv['Female', 'Yes'] / sum(tab_sex_surv[, 'Yes'])
p_female_given_yes_bayes <- p_yes_given_female * p_female / p_yes

data.frame(
  metodo = c('directo', 'bayes'),
  probabilidad = round(c(p_female_given_yes_directa, p_female_given_yes_bayes), 6)
)


## 8) De Bayes a Naive Bayes

Supongamos ahora que queremos predecir una clase `y` a partir de varias variables `x_1, x_2, ..., x_p`.

Bayes dice:

$$P(y \mid x_1, \dots, x_p) = \frac{P(x_1, \dots, x_p \mid y)P(y)}{P(x_1, \dots, x_p)}$$

El problema practico es que modelar la probabilidad conjunta `P(x_1, ..., x_p | y)` puede ser dificil.

**Naive Bayes** simplifica el problema con un supuesto fuerte:

> Las variables predictoras son independientes entre si una vez que conocemos la clase.

Bajo ese supuesto:

$$P(y \mid x_1, \dots, x_p) \propto P(y) \prod_{j=1}^{p} P(x_j \mid y)$$

Eso convierte un problema grande en varios problemas pequenos.


In [ ]:
# Expandir Titanic a nivel fila para usarlo como dataset de clasificacion.
expand_rows <- function(df, count_col = 'Freq') {
  df[rep(seq_len(nrow(df)), df[[count_col]]), setdiff(names(df), count_col), drop = FALSE]
}

titanic_full <- expand_rows(titanic_tab)
titanic_nb <- titanic_full[, c('Class', 'Sex', 'Age', 'Survived')]
titanic_nb[] <- lapply(titanic_nb, factor)

dim(titanic_nb)
head(titanic_nb)


## 9) Entrenar Naive Bayes a mano

Para que el algoritmo sea transparente vamos a implementarlo en R base.

La idea es:

1. Separar train y test.
2. Estimar `P(y)` con las frecuencias de la variable objetivo.
3. Estimar `P(x_j | y)` para cada predictor y cada clase.
4. Multiplicar esas piezas para obtener probabilidades posteriores.
5. Elegir la clase con mayor probabilidad.

Ademas usaremos **suavizado de Laplace** para evitar probabilidades cero.


In [ ]:
# Ajuste manual de un Naive Bayes categorico con suavizado de Laplace.
fit_categorical_nb <- function(df, target, laplace = 1) {
  features <- setdiff(names(df), target)
  y <- factor(df[[target]])
  classes <- levels(y)

  prior_counts <- table(y)
  priors <- (as.numeric(prior_counts) + laplace) /
    (sum(prior_counts) + laplace * length(classes))
  names(priors) <- classes

  likelihoods <- list()
  unknown_probs <- list()

  for (feature in features) {
    x <- factor(df[[feature]])
    tab <- table(x, y)
    feature_levels <- rownames(tab)

    probs <- matrix(
      0,
      nrow = length(feature_levels),
      ncol = length(classes),
      dimnames = list(feature_levels, classes)
    )
    fallback <- setNames(numeric(length(classes)), classes)

    for (cls in classes) {
      counts <- tab[, cls]
      denom <- sum(counts) + laplace * length(feature_levels)
      probs[, cls] <- (counts + laplace) / denom
      fallback[cls] <- laplace / denom
    }

    likelihoods[[feature]] <- probs
    unknown_probs[[feature]] <- fallback
  }

  list(
    target = target,
    features = features,
    classes = classes,
    priors = priors,
    likelihoods = likelihoods,
    unknown_probs = unknown_probs,
    laplace = laplace
  )
}

predict_categorical_nb <- function(model, newdata) {
  newdata <- as.data.frame(newdata, stringsAsFactors = FALSE)
  n <- nrow(newdata)
  log_post <- matrix(
    0,
    nrow = n,
    ncol = length(model$classes),
    dimnames = list(NULL, model$classes)
  )

  for (j in seq_along(model$classes)) {
    cls <- model$classes[j]
    log_post[, j] <- log(model$priors[cls])

    for (feature in model$features) {
      values <- as.character(newdata[[feature]])
      probs <- model$likelihoods[[feature]][values, cls]
      probs[is.na(probs)] <- model$unknown_probs[[feature]][cls]
      log_post[, j] <- log_post[, j] + log(probs)
    }
  }

  log_post_shift <- log_post - apply(log_post, 1, max)
  posterior <- exp(log_post_shift)
  posterior <- posterior / rowSums(posterior)
  pred_class <- model$classes[max.col(posterior, ties.method = 'first')]

  list(
    class = factor(pred_class, levels = model$classes),
    posterior = posterior,
    log_posterior = log_post
  )
}

set.seed(2026)
idx <- sample(seq_len(nrow(titanic_nb)), size = floor(0.75 * nrow(titanic_nb)))
train_nb <- titanic_nb[idx, ]
test_nb <- titanic_nb[-idx, ]

nb_model <- fit_categorical_nb(train_nb, target = 'Survived', laplace = 1)

round(nb_model$priors, 3)
round(nb_model$likelihoods$Sex, 3)
round(nb_model$likelihoods$Class, 3)


## 10) Leer lo que aprendio el modelo

Las salidas anteriores contienen dos tipos de informacion:

- **Priors**: frecuencia de cada clase antes de mirar predictores.
- **Likelihoods**: que tan probable es observar cada valor del predictor dentro de cada clase.

Por ejemplo, una entrada como `P(Sex = Female | Survived = Yes)` ya no es una idea abstracta: es una pieza concreta que el modelo usa para clasificar.

Naive Bayes no "entiende" causalidad. Solo combina frecuencias condicionales de forma sistematica.


In [ ]:
# Prediccion de perfiles hipoteticos.
new_passengers <- data.frame(
  Class = factor(c('1st', '3rd', '2nd'), levels = levels(titanic_nb$Class)),
  Sex = factor(c('Female', 'Male', 'Male'), levels = levels(titanic_nb$Sex)),
  Age = factor(c('Adult', 'Adult', 'Child'), levels = levels(titanic_nb$Age))
)

pred_profiles <- predict_categorical_nb(nb_model, new_passengers)

cbind(
  new_passengers,
  round(as.data.frame(pred_profiles$posterior), 4),
  prediccion = pred_profiles$class
)


## 11) Evaluacion en datos no vistos

Un clasificador debe evaluarse en observaciones que no uso para ajustar probabilidades.

Aqui mediremos:

- **Accuracy**: proporcion total de aciertos.
- **Matriz de confusion**: donde se equivoca y donde acierta.

Accuracy es util, pero nunca debe ser la unica metrica cuando el costo de errores es desigual o las clases estan desbalanceadas.


In [ ]:
# Evaluacion sobre test.
test_pred <- predict_categorical_nb(nb_model, test_nb[, c('Class', 'Sex', 'Age')])
acc_nb <- mean(test_pred$class == test_nb$Survived)
cm_nb <- table(real = test_nb$Survived, pred = test_pred$class)

cat('Accuracy Naive Bayes:', round(acc_nb, 4), '

')
cm_nb

# Algunas probabilidades posteriores de ejemplo.
head(
  cbind(
    test_nb[, c('Class', 'Sex', 'Age', 'Survived')],
    round(as.data.frame(test_pred$posterior), 4),
    prediccion = test_pred$class
  ),
  10
)


## 12) Por que se llama "naive"

El supuesto de independencia condicional casi nunca es exactamente cierto.

En Titanic, por ejemplo, `Class` y `Sex` no son realmente independientes incluso si fijamos la clase objetivo `Survived`.
Aun asi, Naive Bayes suele funcionar sorprendentemente bien porque transforma una estimacion dificil en una aproximacion estable y barata.

La pregunta correcta no es "el supuesto es perfecto?", sino "la aproximacion es suficientemente util para este problema?".


In [ ]:
# Verificar que la independencia condicional no es exacta.
survivors <- subset(train_nb, Survived == 'Yes')

p_joint <- with(survivors, mean(Sex == 'Female' & Class == '1st'))
p_product <- with(survivors, mean(Sex == 'Female') * mean(Class == '1st'))

data.frame(
  medida = c(
    'P(Sex=Female y Class=1st | Survived=Yes)',
    'P(Sex=Female | Yes) * P(Class=1st | Yes)'
  ),
  valor = round(c(p_joint, p_product), 4)
)


## 13) Errores comunes de esta sesion

1. Confundir `P(A|B)` con `P(B|A)`.
2. Suponer independencia sin comprobar si al menos es razonable.
3. Olvidar el suavizado y provocar probabilidades cero.
4. Evaluar el modelo con los mismos datos usados para entrenar.
5. Reportar solo accuracy sin discutir el tipo de error.

Si entiendes por que cada error es peligroso, ya no estas memorizando formulas: estas razonando con probabilidad.


In [ ]:
# Checklist minimo de ideas clave.
checklist <- c(
  'Un evento es un subconjunto del espacio muestral.',
  'Probabilidad condicional cambia el universo de referencia.',
  'Bayes invierte condicionamientos usando prior y evidencia.',
  'Naive Bayes reemplaza una probabilidad conjunta dificil por un producto de probabilidades condicionales.',
  'El supuesto de independencia es fuerte, util y rara vez exacto.'
)

checklist


## 14) Ejercicios para pensar (no copiar codigo)

Antes de programar, escribe tu respuesta en lenguaje natural. Luego, si quieres, valida con R.

1. Si en un problema medico la enfermedad es muy rara, explica por que un test con alta precision puede producir muchos falsos positivos al aplicarse masivamente.
2. En Titanic, compara `P(Survived = Yes | Class = 1st)` con `P(Class = 1st | Survived = Yes)`. Explica por que una puede ser alta sin que la otra tambien lo sea.
3. Disena un ejemplo pequeno donde dos predictores esten fuertemente correlacionados. Explica por que Naive Bayes podria "contar dos veces" evidencia parecida.
4. En el codigo del notebook usamos `Class`, `Sex` y `Age`. Argumenta que variable adicional te gustaria tener y por que podria cambiar la prediccion.
5. Si el accuracy sube de 0.78 a 0.80, que preguntas haria antes de afirmar que el modelo realmente mejoro.
6. Imagina que una categoria nunca aparece en entrenamiento para la clase `Yes`. Explica que pasa sin suavizado de Laplace y por que eso puede ser una mala idea.
7. Propone una situacion real donde prefieras un modelo simple y explicable como Naive Bayes sobre uno mas complejo. Justifica tu decision.


In [ ]:
# Espacio para resolver ejercicios.
# Primero escribe tus argumentos.
# Luego valida con calculos, simulaciones o tablas si hace falta.
